#### QUERY AND VALIDATE `GIZMO.BRONZE.ORDERS`

In [0]:
SELECT *
 FROM gizmo.bronze.orders_vw
LIMIT 30;

#### PARSE ARRAY TYPE FOR `ORDERS`
- EXTRACT AND CAST KEY ORDER FIELDS FROM RAW JSON DATA

In [0]:
SELECT 
  value:order_id::int,
  value:customer_id::int,
  -- value:order_date::date,
  regexp_replace(value, '"order_date": (\d{4}-\d{2}-\d{2})', '"order_date": "\$1"') AS fixed_value,
  value:transaction_timestamp::timestamp,
  value:total_amount::double,
  value:payment_method::string,
  value:items[0].item_id::INT AS item_1,
  value:items[1].item_id::INT AS item_2
 FROM gizmo.bronze.orders_vw;

In [0]:
SELECT 
  get_json_object(value, '$.order_id') AS order_id,
  get_json_object(value, '$.customer_id') AS customer_id,
  get_json_object(value, '$.order_date') AS order_date,
  regexp_replace(value, '"order_date": (\\d{4}-\\d{2}-\\d{2})', '"order_date": "\$1"') AS fixed_value,
  get_json_object(value, '$.transaction_timestamp') AS transaction_timestamp,
  get_json_object(value, '$.total_amount') AS total_amount,
  get_json_object(value, '$.payment_method') AS payment_method,
  get_json_object(value, '$.items[0].item_id') AS first_item_id
FROM gizmo.bronze.orders_vw
LIMIT 10;

In [0]:
SELECT
  CAST(get_json_object(value, '$.order_id') AS INT) AS order_id,
  CAST(get_json_object(value, '$.customer_id') AS INT) AS customer_id,
  CAST(get_json_object(value, '$.order_date') AS DATE) AS order_date,
  regexp_replace(value, '"order_date": (\\d{4}-\\d{2}-\\d{2})', '"order_date": "\$1"') AS fixed_value,
  CAST(get_json_object(value, '$.transaction_timestamp') AS timestamp) AS transaction_timestamp,
  CAST(get_json_object(value, '$.total_amount') AS INT) AS total_amount,
  CAST(get_json_object(value, '$.payment_method') AS STRING) AS payment_method,
  -- Extract and transform items list
  from_json(get_json_object(value, '$.items'), 'array<struct<item_id:int,quantity:int,price:double>>') AS items
FROM
  gizmo.bronze.orders_vw
LIMIT 10;

In [0]:
SELECT
  CAST(get_json_object(value, '$.order_id') AS INT) AS order_id,
  CAST(get_json_object(value, '$.customer_id') AS INT) AS customer_id,
  CAST(get_json_object(value, '$.order_date') AS DATE) AS order_date,
  regexp_replace(value, '"order_date": (\\d{4}-\\d{2}-\\d{2})', '"order_date": "\$1"') AS fixed_value,
  CAST(get_json_object(value, '$.transaction_timestamp') AS timestamp) AS transaction_timestamp,
  CAST(get_json_object(value, '$.total_amount') AS INT) AS total_amount,
  CAST(get_json_object(value, '$.payment_method') AS STRING) AS payment_method,
  from_json(get_json_object(value, '$.items'), 'array<struct<item_id:int,quantity:int,price:double>>') AS items
FROM
  gizmo.bronze.orders_vw
LIMIT 10;

#### WRITE TRANSFORMED DATA TO SILVER SCHEMA
1. CATALOG NAME: GIZMO
2. SCHEMA NAME: SILVER
3. TABLE NAME: ORDERS

In [0]:
CREATE OR REPLACE TEMPORARY VIEW orders_fixed_value_temp_vw
AS
SELECT
regexp_replace(value, '"order_date": (\\d{4}-\\d{2}-\\d{2})', '"order_date": "\$1"') AS fixed_value
FROM
  gizmo.bronze.orders_vw

In [0]:
CREATE OR REPLACE TEMPORARY VIEW orders_vw_temp_vw
AS
SELECT
  from_json(
    fixed_value, 
    'struct<
      items:array<struct<item_id:bigint,name:string,price:bigint,quantity:bigint>>,
      customer_id:int,
      order_date:string,
      order_id:bigint,
      order_status:string,
      payment_method:string,
      total_amount:bigint,
      transaction_timestamp:string
    >'
  ) AS json_value
FROM
  orders_fixed_value_temp_vw;

#### WRITE TRANSFORMED DATA TO SILVER SCHEMA
1. CATALOG NAME: GIZMO
2. SCHEMA NAME: SILVER
3. TABLE NAME: ORDERS

In [0]:
CREATE OR REPLACE TABLE gizmo.silver.orders_json
AS
SELECT * FROM orders_vw_temp_vw;

In [0]:
SELECT
json_value.order_id::int AS order_id,
json_value.customer_id::int AS customer_id,
json_value.order_date::date AS order_date,
json_value.transaction_timestamp::timestamp AS transaction_timestamp,
json_value.total_amount::int AS total_amount,
json_value.payment_method::string AS payment_status,
explode(array_distinct(json_value.items)) AS item
FROM
gizmo.silver.orders_json
ORDER BY 1;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW orders_item_explode_temp_vw
AS
SELECT
json_value.customer_id::INT AS customer_id,
json_value.order_id::INT AS order_id,
json_value.order_date::DATE AS order_date,
json_value.order_status::STRING AS order_status,
json_value.payment_method::STRING AS payment_method,
json_value.total_amount::INT AS total_amount,
json_value.transaction_timestamp::TIMESTAMP AS transaction_timestamp,
explode(array_distinct(json_value.items)) AS item
FROM
gizmo.silver.orders_json;

#### WRITE TRANSFORMED DATA TO SILVER SCHEMA
1. CATALOG NAME: GIZMO
2. SCHEMA NAME: SILVER
3. TABLE NAME: ORDERS


In [0]:
CREATE OR REPLACE TABLE gizmo.silver.orders
AS
SELECT
order_id,
customer_id,
item.item_id,
item.name,
order_date,
order_status,
item.price,
item.quantity,
payment_method,
total_amount,
transaction_timestamp
 FROM orders_item_explode_temp_vw;

#### VALIDATE AND QUERY `GIZMO.SILVER.ORDERS`

In [0]:
SELECT * FROM gizmo.silver.orders LIMIT 15;

In [0]:
%python
dbutils.notebook.exit("ORDERS LOADED INTO GIZMO.SILVER.ORDERS")